In [7]:
import sys, tensorflow as tf
from PIL import Image
import numpy as np
import keras
import os
from keras.applications import MobileNetV2

In [8]:
data_dir = "/Users/ismail/Documents/GitHub/waste-reducing-app-hackathon/images"
test_image = "/Users/ismail/Documents/GitHub/waste-reducing-app-hackathon/test.png"

print("Executable:", sys.executable)
print("TF Version:", tf.__version__)
print(tf.config.list_physical_devices('GPU'))

Executable: /Users/ismail/Documents/GitHub/waste-reducing-app-hackathon/.venv/bin/python
TF Version: 2.18.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [9]:
converted_count = 0
deleted_count = 0

for dir in os.listdir(data_dir):
    if dir == ".DS_Store":
        continue
    for file in os.listdir(os.path.join(data_dir, dir)):
        if file == ".DS_Store":
            continue
        allowed_exts = ["jpeg", "png", "gif", "bmp"]
        try:
            with Image.open(os.path.join(data_dir, dir, file)) as image_loaded:
                image_loaded.verify()

            with Image.open(os.path.join(data_dir, dir, file)) as image_loaded:
                acutal_ext = image_loaded.format.lower()
                if acutal_ext not in allowed_exts:
                    image_loaded.save(
                        os.path.join(data_dir, dir, file.split(".")[0] + ".png"),
                        "PNG",
                        icc_profile=None,
                    )
                    os.remove(os.path.join(data_dir, dir, file))
                    converted_count += 1
                    print(f"Converted {file} to PNG")
                    file = os.path.join(data_dir, dir, file.split(".")[0] + ".png")
                    if image_loaded.mode in ("RGBA", "P", "CMYK"):
                        image_loaded = image_loaded.convert("RGB")
                else:
                    image_loaded.save(
                        os.path.join(data_dir, dir, file),
                        image_loaded.format,
                        icc_profile=None,
                    )
        except Exception as e:
            os.remove(os.path.join(data_dir, dir, file))
            deleted_count += 1
            print(f"Deleted {file} due to error")

print(f"Converted: {converted_count}")
print(f"Deleted: {deleted_count}")

Converted: 0
Deleted: 0


In [10]:
data_augmentation = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.1),
        keras.layers.RandomZoom(0.2),
        keras.layers.RandomFlip("vertical"),
        keras.layers.RandomContrast(0.2),
        keras.layers.RandomTranslation(0.1, 0.1)
    ]
)

2026-11-11 21:07:20.930290: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5
2026-11-11 21:07:20.930321: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2026-11-11 21:07:20.930325: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 12.48 GB
I0000 00:00:1794452840.930364  175017 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1794452840.930396  175017 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [11]:
train_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    image_size=(224, 224),
    batch_size=32,
    seed=42,
)

val_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    image_size=(224, 224),
    batch_size=32,
    seed=42,
)

class_names = train_ds.class_names
num_classes = len(class_names)

Found 2791 files belonging to 17 classes.
Using 2233 files for training.
Found 2791 files belonging to 17 classes.
Using 558 files for validation.


In [12]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet"
)

base_model.trainable = False

In [13]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [14]:
model = keras.Sequential(
    [
        data_augmentation,
        keras.layers.Lambda( 
            lambda x: keras.applications.mobilenet_v2.preprocess_input(x)
        ),
        base_model,
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(num_classes, activation="softmax"),
    ]
)

In [15]:
model.compile(
    loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"]
)

model.fit(train_ds, validation_data=val_ds, epochs=15)

Epoch 1/15


/Users/ismail/Documents/GitHub/waste-reducing-app-hackathon/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-11-11 21:07:23.861622: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


70/70 ━━━━━━━━━━━━━━━━━━━━ 22s 242ms/step - accuracy: 0.2669 - loss: 2.4581 - val_accuracy: 0.5036 - val_loss: 1.7469
Epoch 2/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 14s 205ms/step - accuracy: 0.4810 - loss: 1.6921 - val_accuracy: 0.5681 - val_loss: 1.4890
Epoch 3/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 15s 205ms/step - accuracy: 0.5607 - loss: 1.4260 - val_accuracy: 0.5842 - val_loss: 1.3912
Epoch 4/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 15s 207ms/step - accuracy: 0.5871 - loss: 1.3039 - val_accuracy: 0.6111 - val_loss: 1.3326
Epoch 5/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 15s 205ms/step - accuracy: 0.6296 - loss: 1.2121 - val_accuracy: 0.6147 - val_loss: 1.2942
Epoch 6/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 14s 202ms/step - accuracy: 0.6395 - loss: 1.1360 - val_accuracy: 0.6039 - val_loss: 1.2958
Epoch 7/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 14s 203ms/step - accuracy: 0.6552 - loss: 1.1044 - val_accuracy: 0.6237 - val_loss: 1.2676
Epoch 8/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 14s 201ms/step - accuracy: 0.6650 - loss: 1.0473 - val_accuracy: 0.609

In [16]:
base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False

In [17]:
custom_omtimizer = keras.optimizers.Adam(learning_rate=1e-4)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=custom_omtimizer,
    metrics=["accuracy"]
)

In [18]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    initial_epoch=15,
)

model.save("waste_classification_model.keras")

Epoch 16/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 32s 355ms/step - accuracy: 0.6301 - loss: 1.1964 - val_accuracy: 0.6434 - val_loss: 1.3410
Epoch 17/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 23s 327ms/step - accuracy: 0.7179 - loss: 0.9239 - val_accuracy: 0.6452 - val_loss: 1.4211
Epoch 18/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 23s 332ms/step - accuracy: 0.7600 - loss: 0.7625 - val_accuracy: 0.6523 - val_loss: 1.4076
Epoch 19/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 23s 328ms/step - accuracy: 0.7900 - loss: 0.6614 - val_accuracy: 0.6452 - val_loss: 1.4486
Epoch 20/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 23s 328ms/step - accuracy: 0.8101 - loss: 0.6058 - val_accuracy: 0.6505 - val_loss: 1.4325
Epoch 21/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 23s 326ms/step - accuracy: 0.8361 - loss: 0.5211 - val_accuracy: 0.6344 - val_loss: 1.5036
Epoch 22/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 23s 326ms/step - accuracy: 0.8442 - loss: 0.4871 - val_accuracy: 0.6147 - val_loss: 1.5178
Epoch 23/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 23s 326ms/step - accuracy: 0.8630 - loss: 0.4220 - 